In [1]:
"""
02_traditional_convnets.py / 02_traditional_convnets.ipynb

Self-Contained Benchmark: Geometric Convolutions vs. Standard Normalization in 2D ConvNets
Dataset: CIFAR-10 (45,000 Train / 5,000 Validation / 10,000 Blind Test)
Evaluation: 5 Independent Seeds [42, 1337, 2026, 7, 99], 8 Epochs per Seed

Architectures Evaluated:
  - Vanilla: Standard nn.Conv2d without normalization (baseline)
  - BatchNorm: nn.Conv2d with nn.BatchNorm2d (standard baseline)
  - Hyperspherical: Patch-wise spherical L2 normalization
  - Equatorial: Intra-sample zero-trace projection (S^{d-2}) with fused Weight Standardization

Diagnostics Audited:
  - Test Accuracy and Cross-Entropy Loss
  - Latent Stable Rank (||H||_F^2 / ||H||_2^2)
  - Dead Channel Ratio (% channels with variance < 1e-4)
  - Gradient Signal-to-Noise Ratio (Grad SNR)
  - Loss Landscape Sharpness (Delta L under 2% relative parameter perturbation)
"""

import os
import gc
import time
import pickle
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------------------------------------------------------
# 0. Global Setup and Hardware Configuration
# -----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS: List[int] = [42, 1337, 2026, 7, 99]
EPS: float = 1e-7

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# -----------------------------------------------------------------------------
# 1. Dataset Loader & Preprocessing Pipeline
# -----------------------------------------------------------------------------
def load_cifar10_batch(filepath: str) -> Tuple[np.ndarray, np.ndarray]:
    """Loads a single binary batch of CIFAR-10."""
    with open(filepath, "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    return batch[b"data"], np.array(batch[b"labels"], dtype=np.int64)


def get_cifar10_splits(
    val_size: int = 5000,
    split_seed: int = 42
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Loads and partitions CIFAR-10 into disjoint Train, Validation, and Test sets.
    Normalizes inputs strictly using training split statistics.
    """
    base_path = "/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py"
    if not os.path.exists(base_path):
        base_path = "/kaggle/input/cifar10-python/cifar-10-batches-py"

    if os.path.exists(base_path):
        x_all_list, y_all_list = [], []
        for i in range(1, 6):
            x, y = load_cifar10_batch(os.path.join(base_path, f"data_batch_{i}"))
            x_all_list.append(x)
            y_all_list.append(y)
        x_all = np.concatenate(x_all_list)
        y_all = np.concatenate(y_all_list)
        x_test, y_test = load_cifar10_batch(os.path.join(base_path, "test_batch"))
    else:
        from torchvision import datasets
        train_ds = datasets.CIFAR10(root="./data", train=True, download=True)
        test_ds = datasets.CIFAR10(root="./data", train=False, download=True)
        x_all = train_ds.data.transpose(0, 3, 1, 2).reshape(50000, 3072)
        y_all = np.array(train_ds.targets, dtype=np.int64)
        x_test = test_ds.data.transpose(0, 3, 1, 2).reshape(10000, 3072)
        y_test = np.array(test_ds.targets, dtype=np.int64)

    # Deterministic split: 45,000 train / 5,000 validation
    rng = np.random.RandomState(split_seed)
    indices = rng.permutation(len(x_all))
    val_indices, train_indices = indices[:val_size], indices[val_size:]

    x_train, y_train = x_all[train_indices], y_all[train_indices]
    x_val, y_val = x_all[val_indices], y_all[val_indices]

    # Compute normalization statistics strictly over the training split
    train_mean = np.mean(x_train, axis=0, keepdims=True)
    train_std = np.std(x_train, axis=0, keepdims=True) + 1e-8

    x_train = ((x_train - train_mean) / train_std).reshape(-1, 3, 32, 32).astype(np.float32)
    x_val = ((x_val - train_mean) / train_std).reshape(-1, 3, 32, 32).astype(np.float32)
    x_test = ((x_test - train_mean) / train_std).reshape(-1, 3, 32, 32).astype(np.float32)

    return (
        torch.from_numpy(x_train), torch.from_numpy(y_train),
        torch.from_numpy(x_val), torch.from_numpy(y_val),
        torch.from_numpy(x_test), torch.from_numpy(y_test)
    )


# -----------------------------------------------------------------------------
# 2. Convolutional Primitives
# -----------------------------------------------------------------------------
class HypersphericalConv2d(nn.Module):
    """
    2D convolution with spherical normalization on weights and input receptive fields.
    Computes patch-wise L2 norm using an auxiliary convolution with an all-ones kernel.

    Forward Pass:
        w_norm = W / ||W||_2
        patch_norm = sqrt(conv2d(x^2, 1) + eps)
        y = scale * (conv2d(x, w_norm) / patch_norm)
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1
    ) -> None:
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, kernel_size, kernel_size))
        nn.init.kaiming_normal_(self.weight)
        
        fan_in = in_channels * kernel_size * kernel_size
        self.scale = nn.Parameter(torch.tensor(float(np.sqrt(fan_in))))
        self.register_buffer("ones_kernel", torch.ones(1, in_channels, kernel_size, kernel_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [B, C_in, H, W]
        Returns:
            Output tensor of shape [B, C_out, H_out, W_out]
        """
        w_norm = F.normalize(self.weight, p=2, dim=(1, 2, 3), eps=1e-7)
        x_squared = x ** 2
        patch_norm = torch.sqrt(
            F.conv2d(x_squared, self.ones_kernel, stride=self.stride, padding=self.padding) + 1e-7
        )
        numerator = F.conv2d(x, w_norm, stride=self.stride, padding=self.padding)
        return self.scale * (numerator / patch_norm)


class EquatorialConv2d(nn.Module):
    """
    2D convolution combining intra-sample zero-trace projection (GroupNorm with 1 group)
    with Weight Standardization across spatial and channel dimensions.

    Forward Pass:
        x_eq = (x - mean(x)) / std(x)
        w_cent = W - mean(W)
        w_norm = w_cent / ||w_cent||_2
        y = scale * conv2d(x_eq, w_norm)
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        eps: float = 1e-5
    ) -> None:
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.eps = eps
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, kernel_size, kernel_size))
        nn.init.kaiming_normal_(self.weight)
        
        fan_in = in_channels * kernel_size * kernel_size
        self.scale = nn.Parameter(torch.tensor(float(np.sqrt(fan_in))))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [B, C_in, H, W]
        Returns:
            Output tensor of shape [B, C_out, H_out, W_out]
        """
        x_equatorial = F.group_norm(x, num_groups=1, eps=self.eps)
        w_mean = self.weight.mean(dim=(1, 2, 3), keepdim=True)
        w_centered = self.weight - w_mean
        w_norm = w_centered / (torch.norm(w_centered, p=2, dim=(1, 2, 3), keepdim=True) + self.eps)
        return self.scale * F.conv2d(x_equatorial, w_norm, stride=self.stride, padding=self.padding)


# -----------------------------------------------------------------------------
# 3. Model Architecture Factory
# -----------------------------------------------------------------------------
def build_convnet(layer_type: str = "equatorial", num_classes: int = 10) -> nn.Sequential:
    """
    Constructs a 3-stage ConvNet (3 -> 32 -> 64 -> 128) with specified normalization.
    """
    layers: List[nn.Module] = []

    # Stage 1: 3 -> 32
    if layer_type == "vanilla":
        layers.extend([nn.Conv2d(3, 32, 3, padding=1), nn.GELU()])
    elif layer_type == "batchnorm":
        layers.extend([nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU()])
    elif layer_type == "hyperspherical":
        layers.extend([HypersphericalConv2d(3, 32, 3, padding=1), nn.GELU()])
    elif layer_type == "equatorial":
        layers.extend([EquatorialConv2d(3, 32, 3, padding=1), nn.GELU()])
    else:
        raise ValueError(f"Unknown layer_type: {layer_type}")
    layers.append(nn.MaxPool2d(2, 2))

    # Stage 2: 32 -> 64
    if layer_type == "vanilla":
        layers.extend([nn.Conv2d(32, 64, 3, padding=1), nn.GELU()])
    elif layer_type == "batchnorm":
        layers.extend([nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU()])
    elif layer_type == "hyperspherical":
        layers.extend([HypersphericalConv2d(32, 64, 3, padding=1), nn.GELU()])
    elif layer_type == "equatorial":
        layers.extend([EquatorialConv2d(32, 64, 3, padding=1), nn.GELU()])
    layers.append(nn.MaxPool2d(2, 2))

    # Stage 3: 64 -> 128
    if layer_type == "vanilla":
        layers.extend([nn.Conv2d(64, 128, 3, padding=1), nn.GELU()])
    elif layer_type == "batchnorm":
        layers.extend([nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.GELU()])
    elif layer_type == "hyperspherical":
        layers.extend([HypersphericalConv2d(64, 128, 3, padding=1), nn.GELU()])
    elif layer_type == "equatorial":
        layers.extend([EquatorialConv2d(64, 128, 3, padding=1), nn.GELU()])
    layers.append(nn.AdaptiveAvgPool2d((1, 1)))

    # Classification Head
    layers.append(nn.Flatten())
    layers.append(nn.Linear(128, num_classes))
    return nn.Sequential(*layers)


# -----------------------------------------------------------------------------
# 4. Diagnostic & Health Audit Functions
# -----------------------------------------------------------------------------
def audit_network_health(
    model: nn.Module,
    criterion: nn.Module,
    test_loader: DataLoader,
    train_loader: DataLoader,
    device: torch.device
) -> Tuple[float, float, float, float]:
    """
    Computes structural health metrics:
      1. Stable Rank of penultimate latents: ||H||_F^2 / ||H||_2^2
      2. Dead Channel Ratio: percentage of features with variance < 1e-4
      3. Gradient SNR: norm(mean(grad)) / norm(std(grad)) across minibatches
      4. Loss Landscape Sharpness: Delta L under 2% relative parameter noise
    """
    model.eval()

    # 4.1 Latent Space Audit
    feat_extractor = model[:-1]
    latents_list: List[torch.Tensor] = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            latents_list.append(feat_extractor(xb).cpu())
    latents = torch.cat(latents_list, dim=0)

    # Stable Rank
    h_centered = latents - latents.mean(dim=0, keepdim=True)
    _, s, _ = torch.svd(h_centered)
    srank = (torch.sum(s ** 2) / (torch.max(s) ** 2 + EPS)).item()

    # Dead Channels (%)
    dead_channels = (torch.var(latents, dim=0) < 1e-4).float().mean().item() * 100.0

    # 4.2 Loss Landscape Sharpness (Delta L under 2% weight perturbation)
    base_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            base_loss += criterion(model(xb), yb).item() * yb.size(0)
            total_samples += yb.size(0)
    base_loss /= total_samples

    orig_weights = {}
    with torch.no_grad():
        for name, param in model.named_parameters():
            orig_weights[name] = param.data.clone()
            noise = torch.randn_like(param) * (0.02 * param.data.norm(2) / (param.numel() ** 0.5 + EPS))
            param.data.add_(noise)

    perturbed_loss = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            perturbed_loss += criterion(model(xb), yb).item() * yb.size(0)

    with torch.no_grad():
        for name, param in model.named_parameters():
            param.data.copy_(orig_weights[name])

    sharpness = max(0.0, (perturbed_loss / total_samples) - base_loss)

    # 4.3 Gradient Signal-to-Noise Ratio (SNR)
    model.train()
    batch_grads: List[torch.Tensor] = []
    for idx, (xb, yb) in enumerate(train_loader):
        if idx >= 15:
            break
        xb, yb = xb.to(device), yb.to(device)
        model.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        flat_grads = [p.grad.data.view(-1) for p in model.parameters() if p.grad is not None]
        batch_grads.append(torch.cat(flat_grads))

    grads_stacked = torch.stack(batch_grads)
    mean_g = grads_stacked.mean(dim=0)
    std_g = grads_stacked.std(dim=0) + EPS
    grad_snr = (mean_g.norm(2) / (std_g.norm(2) + EPS)).item()

    return float(srank), float(dead_channels), float(grad_snr), float(sharpness)


# -----------------------------------------------------------------------------
# 5. Benchmark Execution Loop
# -----------------------------------------------------------------------------
def run_benchmark() -> None:
    x_tr, y_tr, x_va, y_va, x_te, y_te = get_cifar10_splits()

    val_loader = DataLoader(TensorDataset(x_va, y_va), batch_size=256, shuffle=False)
    test_loader = DataLoader(TensorDataset(x_te, y_te), batch_size=256, shuffle=False)

    epochs = 8
    lr = 1e-3
    weight_decay = 1e-4

    configs = [
        ("Vanilla", "vanilla"),
        ("BatchNorm", "batchnorm"),
        ("Hyperspherical", "hyperspherical"),
        ("Equatorial", "equatorial")
    ]

    results = {
        cfg[0]: {
            "test_acc": [], "test_loss": [], "srank": [],
            "dead_ch": [], "snr": [], "sharpness": [], "time": []
        }
        for cfg in configs
    }

    print("=" * 115)
    print(f"[INFO] Multi-Seed ConvNet Benchmark | Device: {DEVICE} | Seeds: {len(SEEDS)} | Epochs: {epochs}")
    print("=" * 115)

    for seed_idx, seed in enumerate(SEEDS, 1):
        print(f"\n[INFO] Seed [{seed_idx}/{len(SEEDS)}]: {seed}")
        print("-" * 115)

        g = torch.Generator().manual_seed(seed)
        train_loader = DataLoader(
            TensorDataset(x_tr, y_tr), batch_size=128, shuffle=True, drop_last=True, generator=g
        )

        for name, layer_type in configs:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)

            model = build_convnet(layer_type=layer_type).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = nn.CrossEntropyLoss()

            t0 = time.time()
            for _ in range(1, epochs + 1):
                model.train()
                for xb, yb in train_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    optimizer.zero_grad()
                    loss = criterion(model(xb), yb)
                    loss.backward()
                    optimizer.step()

            elapsed = time.time() - t0

            # Blind Evaluation on Test Split (10,000 samples)
            model.eval()
            test_correct, test_total, test_loss_sum = 0, 0, 0.0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    preds = model(xb)
                    test_loss_sum += criterion(preds, yb).item() * yb.size(0)
                    test_correct += (preds.argmax(dim=-1) == yb).sum().item()
                    test_total += yb.size(0)

            test_acc = (test_correct / test_total) * 100.0
            test_loss = test_loss_sum / test_total

            # Diagnostic Audit
            srank, dead, snr, sharp = audit_network_health(model, criterion, test_loader, train_loader, DEVICE)

            results[name]["test_acc"].append(test_acc)
            results[name]["test_loss"].append(test_loss)
            results[name]["srank"].append(srank)
            results[name]["dead_ch"].append(dead)
            results[name]["snr"].append(snr)
            results[name]["sharpness"].append(sharp)
            results[name]["time"].append(elapsed)

            print(
                f"  [RUN] {name:<18} -> Acc: {test_acc:>6.2f}% | Loss: {test_loss:.4f} | "
                f"Rank: {srank:>5.2f}/128 | SNR: {snr:.3f} | Delta-L: {sharp:.4f} ({elapsed:>4.1f}s)"
            )

            del model, optimizer, criterion
            torch.cuda.empty_cache()

    # -------------------------------------------------------------------------
    # 6. Tabulated Performance Reports
    # -------------------------------------------------------------------------
    print("\n" + "=" * 125)
    print(f"EMPIRICAL PERFORMANCE SUMMARY ({len(SEEDS)} SEEDS, MEAN +/- STD)")
    print("=" * 125)
    print(f"{'ARCHITECTURE':<18} | {'TEST ACC (%)':<16} | {'TEST LOSS':<15} | {'STABLE RANK':<14} | {'GRAD SNR':<12} | {'SHARPNESS':<14} | {'TIME/SEED'}")
    print("-" * 125)
    for name, _ in configs:
        acc_m, acc_s = np.mean(results[name]["test_acc"]), np.std(results[name]["test_acc"])
        loss_m, loss_s = np.mean(results[name]["test_loss"]), np.std(results[name]["test_loss"])
        sr_m, sr_s = np.mean(results[name]["srank"]), np.std(results[name]["srank"])
        snr_m, snr_s = np.mean(results[name]["snr"]), np.std(results[name]["snr"])
        sh_m, sh_s = np.mean(results[name]["sharpness"]), np.std(results[name]["sharpness"])
        t_m = np.mean(results[name]["time"])

        print(
            f"{name:<18} | {acc_m:>6.2f}% +/- {acc_s:<5.2f} | {loss_m:>6.4f} +/- {loss_s:<5.4f} | "
            f"{sr_m:>5.2f} +/- {sr_s:<4.2f} | {snr_m:>5.3f} +/- {snr_s:<4.3f} | {sh_m:>6.4f} +/- {sh_s:<5.4f} | {t_m:>5.1f}s"
        )
    print("=" * 125)


if __name__ == "__main__":
    run_benchmark()

[INFO] Multi-Seed ConvNet Benchmark | Device: cuda | Seeds: 5 | Epochs: 8

[INFO] Seed [1/5]: 42
-------------------------------------------------------------------------------------------------------------------
  [RUN] Vanilla            -> Acc:  64.67% | Loss: 0.9998 | Rank:  2.94/128 | SNR: 1.250 | Delta-L: 0.0000 (13.2s)
  [RUN] BatchNorm          -> Acc:  69.40% | Loss: 0.8669 | Rank:  4.45/128 | SNR: 0.473 | Delta-L: 0.0425 (14.6s)
  [RUN] Hyperspherical     -> Acc:  68.21% | Loss: 0.9061 | Rank:  3.06/128 | SNR: 0.749 | Delta-L: 0.0121 (21.3s)
  [RUN] Equatorial         -> Acc:  70.59% | Loss: 0.8401 | Rank:  4.23/128 | SNR: 1.230 | Delta-L: 0.0099 (16.8s)

[INFO] Seed [2/5]: 1337
-------------------------------------------------------------------------------------------------------------------
  [RUN] Vanilla            -> Acc:  66.25% | Loss: 0.9478 | Rank:  3.43/128 | SNR: 0.584 | Delta-L: 0.0020 (13.1s)
  [RUN] BatchNorm          -> Acc:  68.82% | Loss: 0.8879 | Rank:  4.40